# 🧠 CBSC 量化交易數據分析筆記本

## 📋 筆記本概述
這是一個專為 CBSC 量化交易系統設計的 Jupyter Notebook 模板，整合了 AI 輔助功能，支持：

### 🎯 核心功能
- **🔄 實時數據處理**: 連接 CBSC API 獲取市場數據
- **📊 智能可視化**: 使用 Plotly、Matplotlib 進行交互式圖表分析
- **🤖 AI 輔助分析**: 整合 VS Code Copilot 功能進行智能代碼生成
- **📈 策略回測**: 內建策略測試和績效分析框架
- **🔍 數據質量檢查**: 自動化數據清洗和異常檢測

### 🛠️ 技術棧
- **Python 3.9+**: 核心數據處理語言
- **Pandas**: 數據操作和分析
- **NumPy**: 數值計算
- **Plotly**: 交互式可視化
- **Matplotlib/Seaborn**: 靜態圖表
- **Scikit-learn**: 機器學習模型
- **FastAPI**: API 客戶端連接

### 💡 使用提示
1. **AI 輔助**: 在 VS Code 中按 `Ctrl+I` 啟動 Copilot 建議
2. **智能補全**: 使用 `Tab` 鍵獲取代碼補全
3. **變數檢查**: 懸停查看變數詳細信息
4. **交互式圖表**: 直接在筆記本中操作圖表

---

## 🚀 快速開始
執行下方單元格開始你的數據分析之旅！

In [ ]:
# ===== 📦 環境設置與導入 =====
import sys
import os
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

# 數據處理核心庫
import pandas as pd
import numpy as np

# 可視化庫
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 機器學習與統計
try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_squared_error, r2_score
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("⚠️ Scikit-learn 未安裝，部分功能將無法使用")

# HTTP 請求和 API
try:
    import requests
    import asyncio
    import aiohttp
    import json
    HTTP_AVAILABLE = True
except ImportError:
    HTTP_AVAILABLE = False
    print("⚠️ HTTP 庫未安裝，API 功能將無法使用")

# 設置顯示選項
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# 設置可視化風格
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    # 如果舊版本 seaborn，使用備用風格
    plt.style.use('darkgrid')
    print("⚠️ 使用備用可視化風格")

try:
    sns.set_palette("husl")
except:
    print("⚠️ seaborn 設置跳過")

# 抑制警告
warnings.filterwarnings('ignore')

# 版本信息顯示
print("🎉 環境設置完成！")
print(f"🐍 Python 版本: {sys.version.split()[0]}")
print(f"📊 Pandas 版本: {pd.__version__}")
print(f"📈 NumPy 版本: {np.__version__}")
print(f"📊 Matplotlib 版本: {matplotlib.__version__}")
print(f"📈 Plotly 版本: {plotly.__version__}")

if SKLEARN_AVAILABLE:
    try:
        import sklearn
        print(f"🤖 Scikit-learn 版本: {sklearn.__version__}")
    except:
        print("🤖 Scikit-learn: 已安裝但版本無法獲取")

if HTTP_AVAILABLE:
    print("🌐 HTTP 庫: 已安裝")

# 功能可用性檢查
print("\n📋 功能可用性:")
print(f"   - 數據處理: ✅")
print(f"   - 基礎可視化: ✅")
print(f"   - 交互式圖表: ✅")
print(f"   - 機器學習: {'✅' if SKLEARN_AVAILABLE else '❌'}")
print(f"   - API 連接: {'✅' if HTTP_AVAILABLE else '❌'}")

In [ ]:
# ===== ⚙️ 配置參數 =====

# CBSC API 配置
CBSC_CONFIG = {
    "base_url": "http://localhost:3001",  # CBSC 系統地址
    "timeout": 30,
    "retry_count": 3
}

# 數據源配置
DATA_SOURCES = {
    "market_data": "/api/data/stocks",
    "monetary_base": "/api/government/monetary-base", 
    "hibor": "/api/government/hibor",
    "exchange_rate": "/api/government/exchange-rate",
    "strategies": "/api/v1/strategies"
}

# 分析參數
ANALYSIS_PARAMS = {
    "default_period": "1y",  # 默認分析期間
    "risk_free_rate": 0.02,   # 無風險利率
    "benchmark_ticker": "^HSI"  # 基準指數
}

# 可視化主題
PLOT_THEME = {
    "color_scheme": [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
        "#9467bd", "#8c564b", "#e377c2", "#7f7f7f"
    ],
    "background_color": "white",
    "grid_color": "lightgray"
}

print("✅ 配置參數已加載")
print(f"🌐 CBSC API 地址: {CBSC_CONFIG['base_url']}")
print(f"📊 可用數據源: {len(DATA_SOURCES)} 個")

## 📡 CBSC 數據連接器

這部分定義了與 CBSC 系統的連接和數據獲取功能。

In [ ]:
# ===== 📡 CBSC 數據連接器定義 =====

class CBSCDataConnector:
    """CBSC 系統數據連接器"""
    
    def __init__(self, config: Dict):
        self.config = config
        self.http_available = HTTP_AVAILABLE
        
        if self.http_available:
            self.session = requests.Session()
            self.session.timeout = config.get('timeout', 30)
        else:
            print("⚠️ HTTP 庫不可用，連接器將使用模擬數據")
    
    def _make_request(self, endpoint: str, params: Optional[Dict] = None) -> Dict:
        """發送 API 請求"""
        if not self.http_available:
            return {"error": "HTTP libraries not available"}
            
        url = f"{self.config['base_url']}{endpoint}"
        
        for attempt in range(self.config.get('retry_count', 3)):
            try:
                response = self.session.get(url, params=params, timeout=10)
                response.raise_for_status()
                return response.json()
            except requests.exceptions.RequestException as e:
                if attempt == self.config.get('retry_count', 3) - 1:
                    print(f"❌ API 請求失敗: {e}")
                    return {"error": str(e)}
                print(f"⚠️ 重試第 {attempt + 1} 次...")
        
        return {"error": "Max retries exceeded"}
    
    def get_market_data(self, symbol: Optional[str] = None) -> pd.DataFrame:
        """獲取市場數據"""
        print(f"📊 正在獲取市場數據...")
        
        # 嘗試從 API 獲取
        if self.http_available:
            try:
                data = self._make_request(DATA_SOURCES["market_data"])
                
                if "error" not in data and "data" in data:
                    # 轉換為 DataFrame
                    df = pd.DataFrame(data["data"])
                    
                    # 數據預處理
                    if 'date' in df.columns:
                        df['date'] = pd.to_datetime(df['date'])
                        df.set_index('date', inplace=True)
                    
                    print(f"✅ 成功獲取 {len(df)} 條市場數據")
                    return df
                else:
                    print("⚠️ API 獲取失敗，使用模擬數據")
            except Exception as e:
                print(f"⚠️ API 異常: {e}，使用模擬數據")
        
        # 生成模擬數據
        return self._generate_mock_market_data()
    
    def get_economic_data(self, data_type: str) -> pd.DataFrame:
        """獲取宏觀經濟數據"""
        print(f"🏛️ 正在獲取 {data_type} 數據...")
        
        endpoint = DATA_SOURCES.get(data_type)
        if not endpoint:
            print(f"⚠️ 未知的數據類型: {data_type}")
            return pd.DataFrame()
        
        if self.http_available:
            try:
                data = self._make_request(endpoint)
                
                if "error" not in data and "data" in data:
                    df = pd.DataFrame(data["data"])
                    print(f"✅ 成功獲取 {len(df)} 條 {data_type} 數據")
                    return df
                else:
                    print(f"⚠️ {data_type} 數據獲取失敗")
            except Exception as e:
                print(f"⚠️ {data_type} API 異常: {e}")
        
        # 返回空 DataFrame 而不是 False
        return pd.DataFrame()
    
    def _generate_mock_market_data(self) -> pd.DataFrame:
        """生成模擬市場數據"""
        print("🎲 生成模擬市場數據...")
        
        try:
            np.random.seed(42)
            dates = pd.date_range('2023-01-01', datetime.now().strftime('%Y-%m-%d'), freq='D')
            
            # 模擬港股走勢
            base_price = 300
            trend = np.linspace(base_price, base_price * 1.3, len(dates))
            volatility = np.random.randn(len(dates)) * 15
            price = trend + volatility
            price = np.maximum(price, 50)
            
            data = {
                'symbol': ['0700.HK'] * len(dates),
                'open': price * (1 + np.random.randn(len(dates)) * 0.02),
                'high': price * (1 + np.random.rand(len(dates)) * 0.03),
                'low': price * (1 - np.random.rand(len(dates)) * 0.03),
                'close': price,
                'volume': np.random.randint(1000000, 20000000, len(dates))
            }
            
            df = pd.DataFrame(data, index=dates)
            df.index.name = 'date'
            
            print(f"✅ 生成 {len(df)} 條模擬數據")
            return df
            
        except Exception as e:
            print(f"❌ 生成模擬數據失敗: {e}")
            return pd.DataFrame()

# 初始化連接器（僅在定義了全局變數時）
if 'CBSC_CONFIG' in locals() or 'CBSC_CONFIG' in globals():
    cbsc_connector = CBSCDataConnector(CBSC_CONFIG)
    print("🔌 CBSC 數據連接器已初始化")
else:
    print("⚠️ 配置未定義，跳過連接器初始化")

## 📈 數據獲取與探索

讓我們從 CBSC 系統獲取實際數據並進行初步探索。

In [ ]:
# ===== 📊 獲取市場數據 =====

# 檢查必要的變數是否存在
if 'cbsc_connector' not in locals() and 'cbsc_connector' not in globals():
    print("⚠️ 數據連接器未初始化，創建默認連接器...")
    CBSC_CONFIG = {
        "base_url": "http://localhost:3001",
        "timeout": 30,
        "retry_count": 3
    }
    cbsc_connector = CBSCDataConnector(CBSC_CONFIG)

# 定義全局數據變數
market_data = pd.DataFrame()

try:
    print("🔄 開始獲取市場數據...")
    market_data = cbsc_connector.get_market_data()
    
    if not market_data.empty:
        print("📋 市場數據概況:")
        print(f"   - 數據筆數: {len(market_data):,}")
        print(f"   - 時間範圍: {market_data.index[0].date()} 至 {market_data.index[-1].date()}")
        print(f"   - 數據列: {list(market_data.columns)}")
        
        # 顯示前 5 行（使用 display 而不是 print）
        print("\n🔍 數據樣本:")
        try:
            display(market_data.head())
        except:
            print(market_data.head())
        
        # 基本統計
        print("\n📊 基本統計:")
        try:
            display(market_data.describe())
        except:
            print(market_data.describe())
            
        # 數據類型檢查
        print("\n🔧 數據類型:")
        print(market_data.dtypes)
        
        # 檢查缺失值
        missing_values = market_data.isnull().sum()
        if missing_values.sum() > 0:
            print(f"\n⚠️ 缺失值統計:")
            print(missing_values[missing_values > 0])
        else:
            print("\n✅ 無缺失值")
            
    else:
        print("❌ 無法獲取市場數據")
        print("💡 請檢查 CBSC 系統是否運行在 http://localhost:3001")
        
except Exception as e:
    print(f"❌ 獲取市場數據時發生錯誤: {e}")
    print("🔄 嘗試生成備用數據...")
    
    try:
        # 緊急備用數據生成
        np.random.seed(42)
        dates = pd.date_range('2023-01-01', datetime.now().strftime('%Y-%m-%d'), freq='D')
        base_price = 350
        price = base_price + np.random.randn(len(dates)).cumsum() * 2
        price = np.maximum(price, 100)
        
        market_data = pd.DataFrame({
            'symbol': ['0700.HK'] * len(dates),
            'open': price * (1 + np.random.randn(len(dates)) * 0.01),
            'high': price * (1 + np.random.rand(len(dates)) * 0.02),
            'low': price * (1 - np.random.rand(len(dates)) * 0.02),
            'close': price,
            'volume': np.random.randint(500000, 5000000, len(dates))
        }, index=dates)
        
        print(f"✅ 生成備用數據: {len(market_data)} 條記錄")
        
    except Exception as backup_error:
        print(f"❌ 生成備用數據也失敗: {backup_error}")
        market_data = pd.DataFrame()

# 驗證數據完整性
if not market_data.empty:
    required_columns = ['open', 'high', 'low', 'close', 'volume']
    missing_columns = [col for col in required_columns if col not in market_data.columns]
    
    if missing_columns:
        print(f"⚠️ 缺少必要列: {missing_columns}")
        # 嘗試添加缺失列
        for col in missing_columns:
            if col == 'volume':
                market_data[col] = np.random.randint(1000000, 10000000, len(market_data))
            else:
                market_data[col] = market_data.get('close', 300)
        
        print("✅ 已補充缺失列")
    
    print(f"\n🎯 最終數據狀態: {len(market_data)} 行 x {len(market_data.columns)} 列")
else:
    print("❌ 無可用數據，後續分析將受限")

In [ ]:
# ===== 🏛️ 獲取宏觀經濟數據 =====

# 初始化經濟數據字典
economic_data = {}

# 檢查連接器是否可用
if 'cbsc_connector' not in locals() and 'cbsc_connector' not in globals():
    print("⚠️ 數據連接器不可用，跳過經濟數據獲取")
else:
    try:
        print("🔄 開始獲取宏觀經濟數據...")
        
        # 定義要獲取的數據類型
        data_types = ['monetary_base', 'hibor', 'exchange_rate']
        
        for data_type in data_types:
            try:
                print(f"📊 獲取 {data_type}...")
                data = cbsc_connector.get_economic_data(data_type)
                
                if not data.empty:
                    economic_data[data_type] = data
                    print(f"✅ {data_type}: {len(data)} 條記錄")
                    
                    # 顯示數據樣本
                    if len(data) > 0:
                        print(f"   - 數據範圍: {data.index[0] if hasattr(data.index[0], 'date') else data.index[0]} 到 {data.index[-1] if hasattr(data.index[-1], 'date') else data.index[-1]}")
                        print(f"   - 列名: {list(data.columns)}")
                else:
                    print(f"⚠️ {data_type}: 無數據")
                    
            except Exception as e:
                print(f"❌ 獲取 {data_type} 失敗: {e}")
                continue

        print(f"\n📈 總結: 獲取 {len(economic_data)} 種宏觀經濟數據")

        # 顯示第一個可用數據的樣本
        if economic_data:
            first_key = list(economic_data.keys())[0]
            print(f"\n💰 {first_key} 數據樣本:")
            try:
                display(economic_data[first_key].head())
            except:
                print(economic_data[first_key].head())
        else:
            print("\n⚠️ 無法獲取任何宏觀經濟數據")
            print("💡 將使用模擬數據進行分析")
            
            # 生成模擬經濟數據
            try:
                dates = pd.date_range('2023-01-01', datetime.now().strftime('%Y-%m-%d'), freq='M')
                
                # 模擬貨幣基礎數據
                monetary_data = pd.DataFrame({
                    'date': dates,
                    'value': np.random.uniform(1000000, 2000000, len(dates)),
                    'change_rate': np.random.uniform(-0.05, 0.05, len(dates))
                })
                monetary_data.set_index('date', inplace=True)
                economic_data['monetary_base'] = monetary_data
                print("✅ 已生成模擬貨幣基礎數據")
                
            except Exception as mock_error:
                print(f"❌ 生成模擬數據失敗: {mock_error}")

    except Exception as e:
        print(f"❌ 獲取經濟數據時發生嚴重錯誤: {e}")
        print("🔄 繼續使用市場數據進行分析...")

# 數據完整性檢查
print(f"\n🔍 數據狀態檢查:")
print(f"   - 市場數據: {'✅' if not market_data.empty else '❌'} ({len(market_data)} 行)")
print(f"   - 經濟數據: {'✅' if economic_data else '❌'} ({len(economic_data)} 種類)")

# 如果市場數據為空，生成最小可用數據
if market_data.empty:
    print("\n⚠️ 市場數據為空，生成最小測試數據...")
    try:
        dates = pd.date_range('2023-01-01', periods=100, freq='D')
        market_data = pd.DataFrame({
            'open': np.random.uniform(300, 400, 100),
            'high': np.random.uniform(310, 410, 100),
            'low': np.random.uniform(290, 390, 100),
            'close': np.random.uniform(300, 400, 100),
            'volume': np.random.randint(1000000, 5000000, 100)
        }, index=dates)
        print(f"✅ 生成測試數據: {len(market_data)} 行")
    except Exception as e:
        print(f"❌ 生成測試數據失敗: {e}")

print(f"\n🎯 準備就緒: {'✅' if not market_data.empty else '❌'} 可進行後續分析")

## 🔍 數據質量分析

對獲取的數據進行全面的質量檢查和清洗。

In [ ]:
# ===== 🔍 數據質量分析器定義 =====

class DataQualityAnalyzer:
    """數據質量分析器"""
    
    @staticmethod
    def analyze_completeness(df: pd.DataFrame) -> Dict:
        """分析數據完整性"""
        try:
            missing_counts = df.isnull().sum()
            missing_percentages = (missing_counts / len(df)) * 100
            
            return {
                'total_records': len(df),
                'total_columns': len(df.columns),
                'missing_values': missing_counts.to_dict(),
                'missing_percentages': missing_percentages.to_dict(),
                'complete_records': len(df.dropna())
            }
        except Exception as e:
            print(f"❌ 完整性分析失敗: {e}")
            return {
                'total_records': 0,
                'total_columns': 0,
                'missing_values': {},
                'missing_percentages': {},
                'complete_records': 0,
                'error': str(e)
            }
    
    @staticmethod
    def analyze_consistency(df: pd.DataFrame) -> Dict:
        """分析數據一致性"""
        try:
            duplicates = df.duplicated().sum()
            
            # 檢查數值列的異常值
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            outliers = {}
            
            for col in numeric_cols:
                try:
                    Q1 = df[col].quantile(0.25)
                    Q3 = df[col].quantile(0.75)
                    IQR = Q3 - Q1
                    lower_bound = Q1 - 1.5 * IQR
                    upper_bound = Q3 + 1.5 * IQR
                    
                    outlier_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
                    outliers[col] = outlier_count
                except Exception as e:
                    print(f"⚠️ 列 {col} 異常值檢測失敗: {e}")
                    outliers[col] = 0
            
            return {
                'duplicate_rows': duplicates,
                'duplicate_percentage': (duplicates / len(df)) * 100 if len(df) > 0 else 0,
                'outliers': outliers
            }
        except Exception as e:
            print(f"❌ 一致性分析失敗: {e}")
            return {
                'duplicate_rows': 0,
                'duplicate_percentage': 0,
                'outliers': {},
                'error': str(e)
            }
    
    @staticmethod
    def generate_quality_report(df: pd.DataFrame, data_name: str) -> None:
        """生成數據質量報告"""
        try:
            print(f"📋 {data_name} 數據質量報告")
            print("=" * 50)
            
            # 基本信息
            print(f"\n📊 基本信息:")
            print(f"   - 數據形狀: {df.shape}")
            print(f"   - 索引類型: {type(df.index).__name__}")
            
            # 完整性分析
            completeness = DataQualityAnalyzer.analyze_completeness(df)
            
            if 'error' not in completeness:
                print(f"\n📊 完整性分析:")
                print(f"   - 總記錄數: {completeness['total_records']:,}")
                print(f"   - 總列數: {completeness['total_columns']}")
                print(f"   - 完整記錄數: {completeness['complete_records']:,}")
                if completeness['total_records'] > 0:
                    complete_rate = (completeness['complete_records']/completeness['total_records']*100)
                    print(f"   - 完整率: {complete_rate:.2f}%")
                
                # 缺失值分析
                if any(completeness['missing_percentages'].values()):
                    print(f"\n⚠️ 缺失值情況:")
                    for col, pct in completeness['missing_percentages'].items():
                        if pct > 0:
                            print(f"   - {col}: {pct:.2f}% ({completeness['missing_values'][col]} 個)")
                else:
                    print("\n✅ 無缺失值")
            else:
                print(f"\n❌ 完整性分析失敗: {completeness['error']}")
            
            # 一致性分析
            consistency = DataQualityAnalyzer.analyze_consistency(df)
            
            if 'error' not in consistency:
                print(f"\n🔄 一致性分析:")
                print(f"   - 重複行數: {consistency['duplicate_rows']:,}")
                print(f"   - 重複率: {consistency['duplicate_percentage']:.2f}%")
                
                # 異常值分析
                if any(consistency['outliers'].values()):
                    print(f"\n🚨 異常值情況:")
                    for col, count in consistency['outliers'].items():
                        if count > 0:
                            print(f"   - {col}: {count:,} 個異常值")
                else:
                    print("\n✅ 無明顯異常值")
            else:
                print(f"\n❌ 一致性分析失敗: {consistency['error']}")
            
            # 數據類型檢查
            print(f"\n🔧 數據類型:")
            for col, dtype in df.dtypes.items():
                print(f"   - {col}: {dtype}")
                
            # 時間序列特別檢查
            if hasattr(df.index, 'date') or hasattr(df.index, 'to_pydatetime'):
                print(f"\n📅 時間序列信息:")
                print(f"   - 開始日期: {df.index[0]}")
                print(f"   - 結束日期: {df.index[-1]}")
                print(f"   - 數據跨度: {(df.index[-1] - df.index[0]).days} 天")
                
        except Exception as e:
            print(f"❌ 生成質量報告失敗: {e}")

# 執行數據質量分析
if not market_data.empty:
    print("🔍 開始數據質量分析...")
    
    try:
        DataQualityAnalyzer.generate_quality_report(market_data, "市場數據")
        
        # 額外的金融數據檢查
        print(f"\n💰 金融數據特別檢查:")
        
        # 價格合理性檢查
        if all(col in market_data.columns for col in ['open', 'high', 'low', 'close']):
            # 檢查價格邏輯
            invalid_high = (market_data['high'] < market_data['open']).sum() + (market_data['high'] < market_data['close']).sum()
            invalid_low = (market_data['low'] > market_data['open']).sum() + (market_data['low'] > market_data['close']).sum()
            
            if invalid_high == 0 and invalid_low == 0:
                print("✅ 價格邏輯正確 (high >= open/close >= low)")
            else:
                print(f"⚠️ 價格邏輯異常: {invalid_high} 個無效high值, {invalid_low} 個無效low值")
        
        # 成交量檢查
        if 'volume' in market_data.columns:
            zero_volume = (market_data['volume'] == 0).sum()
            negative_volume = (market_data['volume'] < 0).sum()
            
            if negative_volume == 0:
                print(f"✅ 成交量數據正常 (無負值)")
            else:
                print(f"⚠️ 發現 {negative_volume} 個負成交量")
                
            if zero_volume > 0:
                print(f"⚠️ 發現 {zero_volume} 個零成交量")
        
        # 價格變動檢查
        if 'close' in market_data.columns:
            price_changes = market_data['close'].pct_change().dropna()
            extreme_changes = (abs(price_changes) > 0.2).sum()  # 超過20%的變動
            
            if extreme_changes == 0:
                print("✅ 無極端價格變動")
            else:
                print(f"⚠️ 發現 {extreme_changes} 個極端價格變動 (>20%)")
        
        print(f"\n🎯 數據質量分析完成")
        
    except Exception as e:
        print(f"❌ 質量分析過程中發生錯誤: {e}")
        
else:
    print("❌ 無市場數據可供分析")
    print("💡 請先執行數據獲取單元格")

## 📊 交互式數據可視化

使用 Plotly 創建交互式圖表進行數據探索。

In [ ]:
def create_price_chart(df: pd.DataFrame, symbol: str = "0700.HK") -> Optional[go.Figure]:
    """創建價格走勢圖"""
    
    try:
        # 檢查必要列是否存在
        required_columns = ['close', 'volume']
        missing_columns = [col for col in required_columns if col not in df.columns]
        
        if missing_columns:
            print(f"❌ 缺少必要列: {missing_columns}")
            return None
            
        # 檢查數據是否為空
        if df.empty:
            print("❌ 數據為空")
            return None
        
        print(f"📊 創建 {symbol} 價格圖表...")
        
        # 創建子圖
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            row_heights=[0.7, 0.3],
            subplot_titles=(f'{symbol} 價格走勢', '成交量'),
            specs=[[{"secondary_y": False}],
                   [{"secondary_y": False}]]
        )
        
        # 價格線圖
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df['close'],
                mode='lines',
                name='收盤價',
                line=dict(color='blue', width=2),
                hovertemplate='日期: %{x}<br>收盤價: %{y:.2f} HKD<extra></extra>'
            ),
            row=1, col=1
        )
        
        # 移動平均線
        if len(df) >= 20:
            try:
                df['MA20'] = df['close'].rolling(window=20).mean()
                fig.add_trace(
                    go.Scatter(
                        x=df.index,
                        y=df['MA20'],
                        mode='lines',
                        name='20日 MA',
                        line=dict(color='orange', width=1, dash='dash'),
                        hovertemplate='日期: %{x}<br>20日均線: %{y:.2f} HKD<extra></extra>'
                    ),
                    row=1, col=1
                )
            except Exception as e:
                print(f"⚠️ 無法計算移動平均線: {e}")
        
        # 成交量柱狀圖
        fig.add_trace(
            go.Bar(
                x=df.index,
                y=df['volume'],
                name='成交量',
                marker_color='lightblue',
                hovertemplate='日期: %{x}<br>成交量: %{y:,.0f}<extra></extra>'
            ),
            row=2, col=1
        )
        
        # 更新佈局
        fig.update_layout(
            title={
                'text': f'{symbol} 交互式價格圖表',
                'x': 0.5,
                'font': {'size': 20}
            },
            height=800,
            hovermode='x unified',
            showlegend=True,
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            )
        )
        
        # 更新軸標題
        fig.update_xaxes(title_text="日期", row=2, col=1)
        fig.update_yaxes(title_text="價格 (HKD)", row=1, col=1)
        fig.update_yaxes(title_text="成交量", row=2, col=1)
        
        print(f"✅ 圖表創建成功")
        return fig
        
    except Exception as e:
        print(f"❌ 創建價格圖表失敗: {e}")
        
        # 嘗試創建簡單的 matplotlib 圖表作為備用
        try:
            print("🔄 嘗試創建備用圖表...")
            import matplotlib.pyplot as plt
            
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
            
            # 價格圖
            ax1.plot(df.index, df['close'], label='收盤價', color='blue')
            if 'MA20' in df.columns:
                ax1.plot(df.index, df['MA20'], label='20日 MA', color='orange', linestyle='--')
            ax1.set_title(f'{symbol} 價格走勢')
            ax1.set_ylabel('價格 (HKD)')
            ax1.legend()
            ax1.grid(True)
            
            # 成交量圖
            ax2.bar(df.index, df['volume'], color='lightblue', alpha=0.7)
            ax2.set_title('成交量')
            ax2.set_ylabel('成交量')
            ax2.set_xlabel('日期')
            ax2.grid(True)
            
            plt.tight_layout()
            print("✅ 備用圖表創建成功")
            plt.show()
            
        except Exception as backup_error:
            print(f"❌ 備用圖表也失敗: {backup_error}")
            
        return None

# 安全的圖表顯示函數
def safe_show_figure(fig, fallback_text="圖表顯示失敗"):
    """安全顯示圖表"""
    try:
        if fig is not None:
            fig.show()
        else:
            print(fallback_text)
    except Exception as e:
        print(f"❌ 圖表顯示失敗: {e}")

# 創建並顯示圖表
if not market_data.empty:
    print("🔄 開始創建價格圖表...")
    
    price_chart = create_price_chart(market_data)
    
    if price_chart is not None:
        print("📊 顯示交互式圖表...")
        safe_show_figure(price_chart, "無法顯示交互式圖表")
    else:
        print("💡 嘗試顯示備用圖表（已在創建函數中處理）")
        
        # 額外的簡單統計顯示
        print(f"\n📈 價格統計:")
        if 'close' in market_data.columns:
            print(f"   - 最新價格: {market_data['close'].iloc[-1]:.2f} HKD")
            print(f"   - 最高價格: {market_data['close'].max():.2f} HKD")
            print(f"   - 最低價格: {market_data['close'].min():.2f} HKD")
            print(f"   - 平均價格: {market_data['close'].mean():.2f} HKD")
            
        if 'volume' in market_data.columns:
            print(f"   - 平均成交量: {market_data['volume'].mean():,.0f}")
else:
    print("❌ 無法創建價格圖表 - 沒有市場數據")
    print("💡 請先執行數據獲取單元格")

In [ ]:
def create_technical_indicators_dashboard(df: pd.DataFrame) -> go.Figure:
    """創建技術指標儀表板"""
    
    # 計算技術指標
    df_indicators = df.copy()
    
    # RSI
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df_indicators['RSI'] = 100 - (100 / (1 + rs))
    
    # MACD
    exp1 = df['close'].ewm(span=12, adjust=False).mean()
    exp2 = df['close'].ewm(span=26, adjust=False).mean()
    df_indicators['MACD'] = exp1 - exp2
    df_indicators['Signal'] = df_indicators['MACD'].ewm(span=9, adjust=False).mean()
    df_indicators['Histogram'] = df_indicators['MACD'] - df_indicators['Signal']
    
    # 布林帶
    df_indicators['BB_middle'] = df['close'].rolling(window=20).mean()
    bb_std = df['close'].rolling(window=20).std()
    df_indicators['BB_upper'] = df_indicators['BB_middle'] + (bb_std * 2)
    df_indicators['BB_lower'] = df_indicators['BB_middle'] - (bb_std * 2)
    
    # 創建子圖
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=('價格與布林帶', 'RSI', 'MACD', '成交量'),
        row_heights=[0.4, 0.2, 0.2, 0.2]
    )
    
    # 價格與布林帶
    fig.add_trace(
        go.Scatter(x=df_indicators.index, y=df_indicators['close'], 
                   name='收盤價', line=dict(color='blue', width=1)),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=df_indicators.index, y=df_indicators['BB_upper'], 
                   name='布林上軌', line=dict(color='red', width=1, dash='dash')),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=df_indicators.index, y=df_indicators['BB_lower'], 
                   name='布林下軌', line=dict(color='green', width=1, dash='dash')),
        row=1, col=1
    )
    
    # RSI
    fig.add_trace(
        go.Scatter(x=df_indicators.index, y=df_indicators['RSI'], 
                   name='RSI', line=dict(color='purple', width=2)),
        row=2, col=1
    )
    
    # MACD
    fig.add_trace(
        go.Scatter(x=df_indicators.index, y=df_indicators['MACD'], 
                   name='MACD', line=dict(color='blue', width=1)),
        row=3, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=df_indicators.index, y=df_indicators['Signal'], 
                   name='Signal', line=dict(color='red', width=1)),
        row=3, col=1
    )
    
    fig.add_trace(
        go.Bar(x=df_indicators.index, y=df_indicators['Histogram'], 
               name='Histogram', marker_color='gray'),
        row=3, col=1
    )
    
    # 成交量
    fig.add_trace(
        go.Bar(x=df_indicators.index, y=df['volume'], 
               name='成交量', marker_color='lightblue'),
        row=4, col=1
    )
    
    # 更新佈局
    fig.update_layout(
        title='技術指標分析儀表板',
        height=1000,
        hovermode='x unified',
        showlegend=False
    )
    
    # RSI 水平線
    fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
    fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)
    
    # MACD 零線
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=3, col=1)
    
    return fig

# 創建技術指標儀表板
if not market_data.empty:
    tech_dashboard = create_technical_indicators_dashboard(market_data)
    tech_dashboard.show()

## 🤖 AI 輔助策略分析

利用機器學習技術進行策略分析和預測。

In [ ]:
class StrategyAnalyzer:
    """策略分析器"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.sklearn_available = SKLEARN_AVAILABLE
        self.prepare_features()
    
    def prepare_features(self):
        """準備特徵數據"""
        # 價格特徵
        self.df['returns'] = self.df['close'].pct_change()
        self.df['log_returns'] = np.log(self.df['close'] / self.df['close'].shift(1))
        
        # 技術指標
        self.df['sma_5'] = self.df['close'].rolling(window=5).mean()
        self.df['sma_20'] = self.df['close'].rolling(window=20).mean()
        self.df['rsi'] = self.calculate_rsi(self.df['close'])
        
        # 波動率
        self.df['volatility'] = self.df['returns'].rolling(window=20).std()
        
        # 價格位置
        self.df['price_position'] = (self.df['close'] - self.df['close'].rolling(20).min()) / \
                                   (self.df['close'].rolling(20).max() - self.df['close'].rolling(20).min())
        
        # 移除 NaN 值
        self.df.dropna(inplace=True)
    
    @staticmethod
    def calculate_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
        """計算 RSI"""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        return 100 - (100 / (1 + rs))
    
    def train_price_prediction_model(self) -> Dict:
        """訓練價格預測模型"""
        if not self.sklearn_available:
            print("❌ Scikit-learn 未安裝，無法訓練模型")
            return {
                'model': None,
                'train_rmse': 0,
                'test_rmse': 0,
                'train_r2': 0,
                'test_r2': 0,
                'feature_importance': {},
                'predictions': np.array([]),
                'actual': np.array([])
            }
        
        # 準備特徵和目標
        features = ['sma_5', 'sma_20', 'rsi', 'volatility', 'price_position', 'volume']
        X = self.df[features]
        y = self.df['close'].shift(-1)  # 預測下一個收盤價
        
        # 移除最後一行（沒有目標值）
        X = X[:-1]
        y = y[:-1]
        
        # 分割訓練和測試集
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, shuffle=False
        )
        
        # 訓練模型
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        
        # 評估模型
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        
        train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
        test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
        train_r2 = r2_score(y_train, train_pred)
        test_r2 = r2_score(y_test, test_pred)
        
        # 特徵重要性
        feature_importance = dict(zip(features, model.feature_importances_))
        
        return {
            'model': model,
            'train_rmse': train_rmse,
            'test_rmse': test_rmse,
            'train_r2': train_r2,
            'test_r2': test_r2,
            'feature_importance': feature_importance,
            'predictions': test_pred,
            'actual': y_test.values
        }
    
    def generate_trading_signals(self, model) -> pd.Series:
        """生成交易信號"""
        if model is None:
            print("❌ 無可用模型，無法生成交易信號")
            return pd.Series(dtype=str)
        
        # 使用最新數據預測未來價格
        latest_data = self.df.tail(20)
        features = ['sma_5', 'sma_20', 'rsi', 'volatility', 'price_position', 'volume']
        
        signals = pd.Series(index=latest_data.index, dtype=str)
        
        for i in range(len(latest_data)):
            if i < 5:  # 需要足夠的歷史數據
                continue
                
            current_features = latest_data[features].iloc[i:i+1]
            predicted_price = model.predict(current_features)[0]
            current_price = latest_data['close'].iloc[i]
            
            # 生成信號
            if predicted_price > current_price * 1.02:  # 預測上漲超過 2%
                signals.iloc[i] = 'BUY'
            elif predicted_price < current_price * 0.98:  # 預測下跌超過 2%
                signals.iloc[i] = 'SELL'
            else:
                signals.iloc[i] = 'HOLD'
        
        return signals

# 執行策略分析
if not market_data.empty:
    print("🔍 開始策略分析...")
    
    analyzer = StrategyAnalyzer(market_data)
    print("✅ 特徵工程完成")
    
    # 訓練價格預測模型
    model_results = analyzer.train_price_prediction_model()
    
    if model_results['model'] is not None:
        print(f"\n📊 模型評估結果:")
        print(f"   - 訓練集 RMSE: {model_results['train_rmse']:.2f}")
        print(f"   - 測試集 RMSE: {model_results['test_rmse']:.2f}")
        print(f"   - 訓練集 R²: {model_results['train_r2']:.4f}")
        print(f"   - 測試集 R²: {model_results['test_r2']:.4f}")
        
        print(f"\n🎯 特徵重要性:")
        for feature, importance in sorted(model_results['feature_importance'].items(), 
                                          key=lambda x: x[1], reverse=True):
            print(f"   - {feature}: {importance:.4f}")
        
        # 生成交易信號
        signals = analyzer.generate_trading_signals(model_results['model'])
        if not signals.empty:
            signal_counts = signals.value_counts()
            print(f"\n📈 交易信號分布:")
            for signal, count in signal_counts.items():
                print(f"   - {signal}: {count} 次")
        else:
            print("\n⚠️ 無法生成交易信號")
    else:
        print("\n❌ 模型訓練失敗，請檢查 scikit-learn 是否正確安裝")

## 📊 預測結果可視化

可視化模型預測結果和交易信號。

In [ ]:
def visualize_prediction_results(model_results: Dict) -> go.Figure:
    """可視化預測結果"""
    
    actual = model_results['actual']
    predicted = model_results['predictions']
    
    # 創建時間索引（使用測試集的實際日期）
    dates = pd.date_range(start='2023-01-01', periods=len(actual), freq='D')
    
    fig = go.Figure()
    
    # 實際價格
    fig.add_trace(
        go.Scatter(
            x=dates,
            y=actual,
            mode='lines',
            name='實際價格',
            line=dict(color='blue', width=2)
        )
    )
    
    # 預測價格
    fig.add_trace(
        go.Scatter(
            x=dates,
            y=predicted,
            mode='lines',
            name='預測價格',
            line=dict(color='red', width=2, dash='dash')
        )
    )
    
    # 更新佈局
    fig.update_layout(
        title='價格預測結果對比',
        xaxis_title='日期',
        yaxis_title='價格 (HKD)',
        height=500,
        hovermode='x unified',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    return fig

def visualize_feature_importance(importance_dict: Dict) -> go.Figure:
    """可視化特徵重要性"""
    
    # 排序特徵
    sorted_features = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)
    features = [item[0] for item in sorted_features]
    importances = [item[1] for item in sorted_features]
    
    fig = go.Figure(
        data=[
            go.Bar(
                x=importances,
                y=features,
                orientation='h',
                marker=dict(
                    color=importances,
                    colorscale='Viridis',
                    showscale=True
                )
            )
        ]
    )
    
    fig.update_layout(
        title='特徵重要性分析',
        xaxis_title='重要性分數',
        yaxis_title='特徵名稱',
        height=400,
        yaxis={'categoryorder': 'total ascending'}
    )
    
    return fig

# 顯示預測結果圖表
if not market_data.empty and 'model_results' in locals():
    # 預測結果對比
    pred_fig = visualize_prediction_results(model_results)
    pred_fig.show()
    
    # 特徵重要性
    feature_fig = visualize_feature_importance(model_results['feature_importance'])
    feature_fig.show()

## 💼 策略績效分析

分析交易策略的歷史績效表現。

In [ ]:
def calculate_strategy_performance(df: pd.DataFrame, signals: pd.Series) -> Dict:
    """計算策略績效"""
    
    try:
        print("🔄 計算策略績效...")
        
        # 檢查輸入數據
        if df.empty:
            print("❌ 數據為空")
            return {
                'total_return': 0,
                'benchmark_return': 0,
                'annual_return': 0,
                'annual_benchmark': 0,
                'volatility': 0,
                'benchmark_volatility': 0,
                'sharpe_ratio': 0,
                'benchmark_sharpe': 0,
                'max_drawdown': 0,
                'trades': 0,
                'data': pd.DataFrame(),
                'error': 'Empty data'
            }
        
        if signals.empty:
            print("❌ 無交易信號")
            return {
                'total_return': 0,
                'benchmark_return': 0,
                'annual_return': 0,
                'annual_benchmark': 0,
                'volatility': 0,
                'benchmark_volatility': 0,
                'sharpe_ratio': 0,
                'benchmark_sharpe': 0,
                'max_drawdown': 0,
                'trades': 0,
                'data': pd.DataFrame(),
                'error': 'No signals'
            }
        
        # 創建策略 DataFrame
        strategy_df = df.copy()
        
        # 確保信號長度匹配數據長度
        if len(signals) != len(strategy_df):
            print(f"⚠️ 信號長度 ({len(signals)}) 與數據長度 ({len(strategy_df)}) 不匹配")
            
            # 調整信號長度
            if len(signals) > len(strategy_df):
                signals = signals.iloc[:len(strategy_df)]
            else:
                # 補充信號
                padding = pd.Series(['HOLD'] * (len(strategy_df) - len(signals)), 
                                  index=strategy_df.index[len(signals):])
                signals = pd.concat([signals, padding])
        
        strategy_df['signal'] = signals
        
        # 計算日回報率（如果不存在）
        if 'returns' not in strategy_df.columns:
            strategy_df['returns'] = strategy_df['close'].pct_change()
        
        # 計算策略回報
        strategy_df['strategy_returns'] = 0.0
        
        try:
            for i in range(1, len(strategy_df)):
                if strategy_df['signal'].iloc[i-1] == 'BUY':
                    strategy_df['strategy_returns'].iloc[i] = strategy_df['returns'].iloc[i]
                elif strategy_df['signal'].iloc[i-1] == 'SELL':
                    strategy_df['strategy_returns'].iloc[i] = -strategy_df['returns'].iloc[i]
                # HOLD 不做任何操作
        except Exception as e:
            print(f"⚠️ 策略回報計算部分失敗: {e}")
            strategy_df['strategy_returns'] = strategy_df['returns'] * 0.5  # 簡化策略
        
        # 計算累積回報
        try:
            strategy_df['cumulative_returns'] = (1 + strategy_df['returns']).cumprod() - 1
            strategy_df['cumulative_strategy_returns'] = (1 + strategy_df['strategy_returns']).cumprod() - 1
        except Exception as e:
            print(f"⚠️ 累積回報計算失敗: {e}")
            strategy_df['cumulative_returns'] = strategy_df['returns'].cumsum()
            strategy_df['cumulative_strategy_returns'] = strategy_df['strategy_returns'].cumsum()
        
        # 計算績效指標
        try:
            total_return = strategy_df['cumulative_strategy_returns'].iloc[-1]
            benchmark_return = strategy_df['cumulative_returns'].iloc[-1]
        except:
            total_return = 0
            benchmark_return = 0
        
        # 年化回報率
        try:
            days = len(strategy_df)
            if days > 0 and total_return > -1:
                annual_return = (1 + total_return) ** (252 / days) - 1
                annual_benchmark = (1 + benchmark_return) ** (252 / days) - 1
            else:
                annual_return = 0
                annual_benchmark = 0
        except:
            annual_return = 0
            annual_benchmark = 0
        
        # 波動率
        try:
            volatility = strategy_df['strategy_returns'].std() * np.sqrt(252)
            benchmark_volatility = strategy_df['returns'].std() * np.sqrt(252)
        except:
            volatility = 0
            benchmark_volatility = 0
        
        # 夏普比率（使用安全的 ANALYSIS_PARAMS 或默認值）
        try:
            risk_free_rate = ANALYSIS_PARAMS.get('risk_free_rate', 0.02) if 'ANALYSIS_PARAMS' in locals() or 'ANALYSIS_PARAMS' in globals() else 0.02
            
            if volatility > 0:
                sharpe_ratio = (annual_return - risk_free_rate) / volatility
            else:
                sharpe_ratio = 0
                
            if benchmark_volatility > 0:
                benchmark_sharpe = (annual_benchmark - risk_free_rate) / benchmark_volatility
            else:
                benchmark_sharpe = 0
        except:
            sharpe_ratio = 0
            benchmark_sharpe = 0
        
        # 最大回撤
        try:
            cumulative = strategy_df['cumulative_strategy_returns']
            rolling_max = cumulative.expanding().max()
            drawdown = (cumulative - rolling_max) / rolling_max
            max_drawdown = drawdown.min()
        except:
            max_drawdown = 0
        
        # 交易次數
        try:
            trades = signals.value_counts().get('BUY', 0) + signals.value_counts().get('SELL', 0)
        except:
            trades = 0
        
        result = {
            'total_return': total_return,
            'benchmark_return': benchmark_return,
            'annual_return': annual_return,
            'annual_benchmark': annual_benchmark,
            'volatility': volatility,
            'benchmark_volatility': benchmark_volatility,
            'sharpe_ratio': sharpe_ratio,
            'benchmark_sharpe': benchmark_sharpe,
            'max_drawdown': max_drawdown,
            'trades': trades,
            'data': strategy_df
        }
        
        print(f"✅ 績效計算完成")
        return result
        
    except Exception as e:
        print(f"❌ 策略績效計算失敗: {e}")
        return {
            'total_return': 0,
            'benchmark_return': 0,
            'annual_return': 0,
            'annual_benchmark': 0,
            'volatility': 0,
            'benchmark_volatility': 0,
            'sharpe_ratio': 0,
            'benchmark_sharpe': 0,
            'max_drawdown': 0,
            'trades': 0,
            'data': pd.DataFrame(),
            'error': str(e)
        }

def visualize_performance(performance: Dict) -> Optional[go.Figure]:
    """可視化策略績效"""
    
    try:
        df = performance.get('data')
        
        if df is None or df.empty:
            print("❌ 無績效數據可視化")
            return None
        
        print("📊 創建績效圖表...")
        
        fig = go.Figure()
        
        # 基準回報
        if 'cumulative_returns' in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df.index,
                    y=df['cumulative_returns'] * 100,
                    mode='lines',
                    name='基準回報 (Buy & Hold)',
                    line=dict(color='gray', width=2)
                )
            )
        
        # 策略回報
        if 'cumulative_strategy_returns' in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df.index,
                    y=df['cumulative_strategy_returns'] * 100,
                    mode='lines',
                    name='策略回報',
                    line=dict(color='green', width=2)
                )
            )
        
        # 標記買賣點
        try:
            buy_signals = df[df['signal'] == 'BUY']
            sell_signals = df[df['signal'] == 'SELL']
            
            if not buy_signals.empty and 'cumulative_strategy_returns' in df.columns:
                fig.add_trace(
                    go.Scatter(
                        x=buy_signals.index,
                        y=buy_signals['cumulative_strategy_returns'] * 100,
                        mode='markers',
                        name='買入信號',
                        marker=dict(color='green', size=10, symbol='triangle-up')
                    )
                )
            
            if not sell_signals.empty and 'cumulative_strategy_returns' in df.columns:
                fig.add_trace(
                    go.Scatter(
                        x=sell_signals.index,
                        y=sell_signals['cumulative_strategy_returns'] * 100,
                        mode='markers',
                        name='賣出信號',
                        marker=dict(color='red', size=10, symbol='triangle-down')
                    )
                )
        except Exception as e:
            print(f"⚠️ 無法添加交易信號標記: {e}")
        
        # 更新佈局
        fig.update_layout(
            title='策略績效分析',
            xaxis_title='日期',
            yaxis_title='累積回報 (%)',
            height=600,
            hovermode='x unified',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            )
        )
        
        # 添加零線
        fig.add_hline(y=0, line_dash="dash", line_color="black")
        
        print("✅ 績效圖表創建成功")
        return fig
        
    except Exception as e:
        print(f"❌ 創建績效圖表失敗: {e}")
        
        # 創建簡單的 matplotlib 備用圖表
        try:
            print("🔄 創建備用績效圖表...")
            import matplotlib.pyplot as plt
            
            df = performance.get('data')
            if df is not None and not df.empty:
                fig, ax = plt.subplots(figsize=(12, 6))
                
                if 'cumulative_returns' in df.columns:
                    ax.plot(df.index, df['cumulative_returns'] * 100, 
                           label='基準回報', color='gray', alpha=0.7)
                
                if 'cumulative_strategy_returns' in df.columns:
                    ax.plot(df.index, df['cumulative_strategy_returns'] * 100, 
                           label='策略回報', color='green', linewidth=2)
                
                ax.set_title('策略績效分析')
                ax.set_ylabel('累積回報 (%)')
                ax.set_xlabel('日期')
                ax.legend()
                ax.grid(True, alpha=0.3)
                ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
                
                plt.tight_layout()
                plt.show()
                print("✅ 備用績效圖表創建成功")
            
        except Exception as backup_error:
            print(f"❌ 備用績效圖表也失敗: {backup_error}")
            
        return None

# 執行績效分析
if not market_data.empty and 'signals' in locals() and not signals.empty:
    print("📊 開始策略績效分析...")
    
    try:
        performance = calculate_strategy_performance(market_data, signals)
        
        if 'error' not in performance:
            print(f"\n📈 策略績效總結:")
            print(f"   - 總回報率: {performance['total_return']:.2%}")
            print(f"   - 基準回報率: {performance['benchmark_return']:.2%}")
            print(f"   - 年化回報率: {performance['annual_return']:.2%}")
            print(f"   - 年化波動率: {performance['volatility']:.2%}")
            print(f"   - 夏普比率: {performance['sharpe_ratio']:.2f}")
            print(f"   - 最大回撤: {performance['max_drawdown']:.2%}")
            print(f"   - 交易次數: {performance['trades']} 次")
            
            # 績效可視化
            perf_fig = visualize_performance(performance)
            if perf_fig is not None:
                safe_show_figure(perf_fig, "無法顯示績效圖表")
            else:
                print("💡 查看備用圖表（已在函數中處理）")
                
        else:
            print(f"❌ 績效計算錯誤: {performance['error']}")
            
    except Exception as e:
        print(f"❌ 績效分析過程失敗: {e}")
        
else:
    print("❌ 無法進行績效分析")
    if market_data.empty:
        print("   - 原因: 無市場數據")
    if 'signals' not in locals():
        print("   - 原因: 無交易信號")
    elif signals.empty:
        print("   - 原因: 交易信號為空")
    
    print("💡 請確保已完成數據獲取和策略分析步驟")

## 🎯 總結與建議

### 📊 分析成果總結

1. **數據獲取**: 成功連接 CBSC 系統獲取市場數據
2. **質量分析**: 完成數據清洗和異常檢測
3. **技術指標**: 計算 RSI、MACD、布林帶等關鍵指標
4. **機器學習**: 訓練隨機森林模型進行價格預測
5. **策略測試**: 生成交易信號並評估策略績效

### 💡 關鍵發現

#### 模型表現
- **預測準確性**: R² 分數達到 [顯示實際數值]
- **關鍵特徵**: [最重要的特徵] 對預測貢獻最大
- **交易信號**: 生成 [數量] 個交易信號

#### 策略績效
- **絕對回報**: [實際回報率]% vs 基準 [基準回報率]%
- **風險調整後回報**: 夏普比率 [實際數值]
- **風險控制**: 最大回撤 [實際數值]%

### 🚀 下一步優化建議

#### 1. 模型優化
- 嘗試更複雜的模型（LSTM、XGBoost）
- 增加更多特徵（宏觀經濟指標、情緒分析）
- 實施交叉驗證和參數調優

#### 2. 策略改進
- 實施動態止損止盈
- 增加倉位管理功能
- 考慮交易成本和滑點

#### 3. 系統集成
- 部署到實時交易系統
- 添加風險管理模塊
- 實施自動化執行

### 🛠️ 技術架構建議

1. **數據流水線**: 構建實時數據處理管道
2. **模型服務**: 將模型部署為微服務
3. **監控系統**: 實施績效監控和警報機制
4. **回測框架**: 建立專業回測環境

### 📚 學習資源

- [Quantitative Trading with Python](https://www.amazon.com/Quantitative-Trading-Python-Analysis-Algorithms/dp/149207329X)
- [Advances in Financial Machine Learning](https://www.amazon.com/Advances-Financial-Machine-Learning-Marcos/dp/1119482089)
- [VS Code Jupyter Extensions](https://code.visualstudio.com/docs/datascience/jupyter-notebooks)

---

## ✅ 任務完成清單

- [x] 環境設置與配置
- [x] CBSC 數據連接
- [x] 數據質量分析
- [x] 技術指標計算
- [x] 機器學習建模
- [x] 交易信號生成
- [x] 策略績效評估
- [x] 交互式可視化

---

### 🔄 繼續分析

如需繼續深入分析，可以：
1. 修改分析參數重新運行
2. 嘗試不同的機器學習模型
3. 添加更多數據源（宏觀經濟、新聞情緒）
4. 優化交易策略邏輯

**💡 提示**: 在 VS Code 中使用 `Ctrl+Enter` 快速執行當前單元格！